# Laboration 1

## imports

In [170]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import data_loading_code
from torch.utils.data import TensorDataset, DataLoader

import wandb

## Load Data

Might be wise to not load the 25K file if it isn't needed then only load the regular one.

In [171]:
from importlib import reload
reload(data_loading_code)
from data_loading_code import load_data#, load_data_LSTM

train_x, train_y, val_x, val_y, vocab_size = load_data("amazon_cells_labelled.txt")
# train_L_x, train_L_y, val_L_x, val_L_y = load_data("amazon_cells_labelled_LARGE_25K.txt")
# print(train_x.shape)

# print(train_L_x)

train_dataset_LSTM = TensorDataset(train_x, train_y)
train_loader_LSTM = DataLoader(train_dataset_LSTM, batch_size=32, shuffle=True)

val_dataset_LSTM = TensorDataset(val_x, val_y)
val_loader_LSTM = DataLoader(val_dataset_LSTM, batch_size=32)

c:\Users\Rasmus\OneDrive\Dokumente\D0012E\ANN-and-Transformers\ANN-and-Transformers\data_loading_code.py:23: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rasmus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [172]:
input_size = train_x.shape[1]

model_1_1 = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Dropout(0.15),
    nn.Linear(128,2) # Negative or positive review
)

class LSTM(nn.Module):
    def __init__(self, input_size = vocab_size, embed_dim = 100, hidden_size = 64):
        super().__init__()
        self.embedding = nn.Embedding(input_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first= True)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

## Task 1.1.1
A simple neural network composed of linear layers. You may incorporate activation
functions, dropout, and other complementary layers as needed.

In [ ]:
wandb.init(project="Task 1.1.1: Simple ANN", name="ANN-1")


wandb.finish()

## Task 1.1.2
A neural network based on LSTM or bidirectional LSTM (Bi-LSTM) layers.

In [177]:
# wandb.init(project="Task 1.1.2: LSTM RNN", name="RNN-1")

model_LSTM = LSTM()
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model_LSTM.parameters(), lr=0.0001)
optimizer = torch.optim.Adam(model_LSTM.parameters(), lr=0.001)
num_epochs = 10

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor = 0.1,
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')


for epoch in range(num_epochs):
    model_LSTM.train()

    average_train_loss = 0.0
    for batch_x, batch_y in train_loader_LSTM:
#------------------------- TRAINING -------------------------
        optimizer.zero_grad()
        output = model_LSTM(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        average_train_loss += loss.item()/len(train_loader_LSTM)

#------------------------- VALIDATION -------------------------
    model_LSTM.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader_LSTM:
                output = model_LSTM(batch_x)
                val_loss = criterion(output, batch_y)
                val_running_loss += val_loss.item()
        average_val_loss = val_running_loss/len(val_loader_LSTM)
    scheduler.step(average_val_loss)

#------------------------- VISUALIZATION -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_train_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )
#------------------------- SAVING BEST MODEL -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model_LSTM.state_dict(), "BestModel_LSTM.pth")
        print("Saved The Best Performing Model")


# wandb.finish()

Epoch [1/10] train loss: 0.693, val loss: 0.696, lr: 0.001000
Saved The Best Performing Model
Epoch [2/10] train loss: 0.693, val loss: 0.696, lr: 0.001000
Saved The Best Performing Model
Epoch [3/10] train loss: 0.692, val loss: 0.695, lr: 0.001000
Saved The Best Performing Model
Epoch [4/10] train loss: 0.689, val loss: 0.697, lr: 0.001000
Epoch [5/10] train loss: 0.648, val loss: 0.685, lr: 0.001000
Saved The Best Performing Model
Epoch [6/10] train loss: 0.476, val loss: 0.649, lr: 0.001000
Saved The Best Performing Model
Epoch [7/10] train loss: 0.361, val loss: 0.657, lr: 0.001000
Epoch [8/10] train loss: 0.307, val loss: 0.801, lr: 0.001000
Epoch [9/10] train loss: 0.242, val loss: 0.755, lr: 0.001000
Epoch [10/10] train loss: 0.218, val loss: 0.814, lr: 0.000100


In [178]:
#Test the model
model_LSTM.load_state_dict(torch.load("BestModel_LSTM.pth"))
model_LSTM.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in val_loader_LSTM:
        output = model_LSTM(batch_x)
        val_loss = criterion(output, batch_y)
        test_loss += val_loss.item()

        HighestValue, predicted = torch.max(output, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

average_test_loss = test_loss / len(val_loader_LSTM)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

wandb.finish()

Test loss: 0.649
Test accuracy: 68.00%


## Task 1.2: Transformers Implementation
For this task, you will implement your transformer in PyTorch. 

In [ ]:
# rekommenderade paket från länken
!pip install tqdm boto3 requests regex sentencepiece sacremoses

## Task 1.3 Comparison
Here, you should compare of all three models; you are requested to use the same test dataset
for Simple ANN, LSTM based model and the Transformer to answer the following:

• Compare the performance of the two models and explain in which scenarios you would
prefer one over the other.

• How did the two models’ complexity, accuracy, and efficiency differ? Did one model
outperform the other in specific scenarios or tasks? If so, why?

• What insights did you obtain concerning data amount to train? Embedding utilized?
Architectural choices made?